# 01 Prevent Eda

Exploratory data analysis notebook (Python)

Purpose: inspect distributions, incidence patterns, and intermediate risk calculations before the fairness analyses.

# Set Up Environment

In [ ]:
# import client module with default PySpark output mode
from truveta.study import Client, OutputMode, display_df
client = Client()

# import client module and set PandasOnSpark output mode
from truveta.study import Client, OutputMode, display_df
client = Client(output_mode = OutputMode.PandasOnSpark)

In [ ]:
# get study
study = client.get_study()

In [ ]:
packages = ["autograd", "autograd_gamma", "formulaic", "interface_meta", "lifelines", "pandas", "tzdata"]
study.install_packages(packages)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter

In [ ]:
saved_df = study.load_artifacts_data("/data/COHORT_SDOH_DATA_PREVENT_PCE_01_16_2025_w_Insurance.parquet")
display_df(saved_df)

In [ ]:
df = saved_df.to_pandas()

# Visualization 

In [ ]:
def plot_distribution(df):
    # List of variables to plot
    variables = ['age','scre', 'sbp', 'bmi', 'chol', 'hdlc', 'fu_mi', 'fu_str', 'fu_hf']
    
    # Create a grid of subplots
    fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 12))
    axes = axes.flatten()
    
    # Plot each variable's distribution
    for i, var in enumerate(variables):
        sns.histplot(df[var], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {var}')
    
    # Remove the last empty subplot (if there is an odd number of variables)
    if len(variables) % 2 != 0:
        fig.delaxes(axes[-1])

    plt.tight_layout()
    plt.show()

plot_distribution(df) 

In [ ]:
def plot_binary_variable_frequencies_with_percentage(df):
    # List of binary variables
    binary_vars = ['female', 'diabetes', 'cursmk', 'antihtn', 'statin', 
                   'event_mi', 'event_str', 'event_hf', 'event_death']
    
    # Create a grid of subplots
    fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(14, 12))
    axes = axes.flatten()
    
    # Plot frequency of each binary variable with percentage
    for i, var in enumerate(binary_vars):
        count_data = df[var].value_counts(normalize=True) * 100
        sns.barplot(x=count_data.index, y=count_data.values, ax=axes[i])
        axes[i].set_title(f'Frequency of {var}')
        axes[i].set_xlabel(f'{var} (0 = No, 1 = Yes)')
        axes[i].set_ylabel('Percentage')
        for p in axes[i].patches:
            axes[i].annotate(f'{p.get_height():.1f}%', 
                             (p.get_x() + p.get_width() / 2., p.get_height()), 
                             ha='center', va='center', fontsize=10, color='black', xytext=(0, 5), 
                             textcoords='offset points')

    plt.tight_layout()
    plt.show()
plot_binary_variable_frequencies_with_percentage(df)

## Investigating Spike

In [ ]:
filtered_data = df[(df['hdlc'] >= 40) & (df['hdlc'] <= 50)]

# Get the count of each unique entry
unique_counts = filtered_data['hdlc'].value_counts()

# Display the result
print(unique_counts)

# Data Cleaning 

In [ ]:
# Convert index_dtt to year
df['year'] = pd.to_datetime(df['index_dtt']).dt.year

In [ ]:
df['scre'] = df['scre'].astype(float)
df['age'] = df['age'].astype(float)

In [ ]:
scre = df['scre'].to_numpy()
age = df['age'].to_numpy()
female = df['female'].to_numpy()

In [ ]:
egfr_female = 141.57 * np.minimum(scre / 0.7, 1) ** -0.241 * np.maximum(scre / 0.7, 1) ** -1.2 * 0.993839 ** age
egfr_male = 141.57 * np.minimum(scre / 0.9, 1) ** -0.302 * np.maximum(scre / 0.9, 1) ** -1.2 * 0.993839 ** age
df['egfr'] = np.where(female == 0, egfr_male, egfr_female)

In [ ]:
df['fu_ascvd'] = np.minimum(df['fu_mi'], df['fu_str'])
df['fu_cvd'] = np.minimum.reduce([df['fu_mi'], df['fu_str'], df['fu_hf']])

In [ ]:
df['age'] = (df['age'] - 55) / 10
df['nonhdlc'] = (df['chol'] - df['hdlc']) * 0.02586 - 3.5
df['hdlc'] = (df['hdlc'] * 0.02586 - 1.3) / 0.3
df['sbp1'] = (np.minimum(df['sbp'], 110) - 110) / 20
df['sbp2'] = (np.maximum(df['sbp'], 110) - 130) / 20
df['bmi1'] = (np.minimum(df['bmi'], 30) - 25) / 5
df['bmi2'] = (np.maximum(df['bmi'], 30) - 30) / 5
df['egfr1'] = (np.minimum(df['egfr'], 60) - 60) / -15
df['egfr2'] = (np.maximum(df['egfr'], 60) - 90) / -15

In [ ]:
df['sbp2treat'] = df['sbp2'] * df['antihtn']
df['nonhdlctreat'] = df['nonhdlc'] * df['statin']
df['agenonhdlc'] = df['age'] * df['nonhdlc']
df['agehdlc'] = df['age'] * df['hdlc']
df['agesbp2'] = df['age'] * df['sbp2']
df['agediabetes'] = df['age'] * df['diabetes']
df['agecursmk'] = df['age'] * df['cursmk']
df['agebmi2'] = df['age'] * df['bmi2']
df['ageegfr1'] = df['age'] * df['egfr1']
df['cons'] = 1

## Check Incidence Rate in a Sliding Window

In [ ]:
# Define the three-year intervals
intervals = [(2010, 2012), (2011, 2013), (2012, 2014), (2013, 2015), 
             (2014, 2016), (2015, 2017), (2016, 2018)]

# Create a dictionary to store the results
proportions = {}

# Loop through each interval
for start_year, end_year in intervals:
    # Filter the dataframe for the current interval
    interval_df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    
    # Calculate the proportion of each event in the interval
    total = len(interval_df)
    if total > 0:
        mi_prop = interval_df['event_mi'].mean()  # Proportion of MI events
        str_prop = interval_df['event_str'].mean()  # Proportion of stroke events
        hf_prop = interval_df['event_hf'].mean()  # Proportion of heart failure events
    else:
        mi_prop, str_prop, hf_prop = 0, 0, 0
    
    # Store the proportions in the dictionary
    proportions[f'{start_year}-{end_year}'] = {
        'mi_proportion': mi_prop,
        'stroke_proportion': str_prop,
        'hf_proportion': hf_prop
    }

# Convert the dictionary to a DataFrame for easy viewing
proportions_df = pd.DataFrame(proportions).T
proportions_df

In [ ]:
# Plot the proportions
plt.figure(figsize=(10, 6))

# Extract the x-axis labels (intervals)
x_labels = proportions_df.index

# Plot MI proportion
plt.plot(x_labels, proportions_df['mi_proportion'], label='MI Proportion', marker='o')

# Plot Stroke proportion
plt.plot(x_labels, proportions_df['stroke_proportion'], label='Stroke Proportion', marker='o')

# Plot Heart Failure proportion
plt.plot(x_labels, proportions_df['hf_proportion'], label='HF Proportion', marker='o')

# Add labels and title
plt.xlabel('Three-Year Intervals')
plt.ylabel('Proportion')
plt.title('Proportion of MI, Stroke, and Heart Failure Events Over Time')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Add legend
plt.legend()

# Show plot
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(df['hdlc'], bins=30, kde=True)
plt.title('hdlc After Transformation')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

# Risk from Prevent Coefficient

## CVD Risk

### Male

In [ ]:
# Coefficient matrix for males
coefficient_male_cvd = np.array([
0.7688528,0.0736174,
 -0.0954431,
 -0.4347345,
 0.3362658,
 0.7692857,
 0.4386871,
 0.5378979,
 0.0164827,
 0.288879,
 -0.1337349,
 -0.0475924,
 0.150273,
 -0.0517874,
 0.0191169,
 -0.1049477,
 -0.2251948,
 -0.0895067,
 -0.1543702,-3.031168
]).reshape(20, 1)

# Subset the data for males (female == 0)
covariate_male_cvd = df[(df['female'] == 0) ][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_male_cvd = np.dot(covariate_male_cvd, coefficient_male_cvd)

# Calculate the risk
risk_male_cvd = np.exp(lo_male_cvd) / (1 + np.exp(lo_male_cvd))

### Female

In [ ]:
# Coefficient matrix for females
coefficient_female_cvd = np.array([ 0.7939329, 0.0305239, -0.1606857, -0.2394003, 0.360078, 0.8667604,  0.5360739, 0.6045917, 0.0433769, 0.3151672, -0.1477655, -0.0663612, 0.1197879,  -0.0819715, 0.0306769, -0.0946348, -0.27057, -0.078715, -0.1637806, -3.307728]).reshape(20, 1)

# Subset the data for females (female == 1) 
covariate_female_cvd = df[(df['female'] == 1)][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_female_cvd = np.dot(covariate_female_cvd, coefficient_female_cvd)

# Calculate the risk (probability)
risk_female_cvd = np.exp(lo_female_cvd) / (1 + np.exp(lo_female_cvd))

## ASCVD Risk

### Male

In [ ]:
# Coefficient matrix for males
coefficient_male_ascvd = np.array([
    0.7099847198, 0.1658663005, -0.1144284979, -0.2837212086, 0.3239977062, 0.7189596891,
    0.3956972957, 0.369007498, 0.0203619003, 0.2036522031, -0.0865581036, -0.0322915986,
    0.1145630032, -0.0300005004, 0.0232746992, -0.0927024037, -0.2018525004, -0.0970527008,
    -0.1217081025, -3.500654936
]).reshape(20, 1)

# Subset the data for males (female == 0)
covariate_male_ascvd = df[(df['female'] == 0) ][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_male_ascvd = np.dot(covariate_male_ascvd, coefficient_male_ascvd)

# Calculate the risk
risk_male_ascvd = np.exp(lo_male_ascvd) / (1 + np.exp(lo_male_ascvd))

### Female

In [ ]:
# Coefficient matrix for females
coefficient_female_ascvd = np.array([
    0.7198830247, 0.1176967025, -0.1511850059, -0.0835357979, 0.3592852056, 0.8348584771,
    0.4831078053, 0.4864619076, 0.039777901, 0.2265308946, -0.0592373982, -0.0395761989,
    0.0844423026, -0.0567838997, 0.0325691998, -0.1035984978, -0.241754204, -0.0791141987,
    -0.167149201, -3.819974899
]).reshape(20, 1)

# Subset the data for females (female == 1) 
covariate_female_ascvd = df[(df['female'] == 1)][[
    'age', 'nonhdlc', 'hdlc', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'egfr1', 'egfr2',
    'antihtn', 'statin', 'sbp2treat', 'nonhdlctreat', 'agenonhdlc', 'agehdlc', 'agesbp2',
    'agediabetes', 'agecursmk', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_female_ascvd = np.dot(covariate_female_ascvd, coefficient_female_ascvd)

# Calculate the risk (probability)
risk_female_ascvd = np.exp(lo_female_ascvd) / (1 + np.exp(lo_female_ascvd))

## Heart Failure

### Male

In [ ]:
# Coefficient matrix for males (Heart Failure)
coefficient_male_hf = np.array([
    0.8972641826, -0.6811466217, 0.3634460866, 0.9237759709, 0.5023735762, -0.0485840999,
    0.3726929128, 0.6926916838, 0.0251826998, 0.2980921865, -0.0497731008, -0.1289200932,
    -0.3040924072, -0.140168801, 0.0068126, -0.179777801, -3.946391106
]).reshape(17, 1)

# Subset the data for males (female == 0) 
covariate_male_hf = df[(df['female'] == 0)][[
    'age', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'bmi1', 'bmi2', 'egfr1', 'egfr2',
    'antihtn', 'sbp2treat', 'agesbp2', 'agediabetes', 'agecursmk', 'agebmi2', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_male_hf = np.dot(covariate_male_hf, coefficient_male_hf)

# Calculate the risk (probability)
risk_male_hf = np.exp(lo_male_hf) / (1 + np.exp(lo_male_hf))

### Female 

In [ ]:
# Coefficient matrix for females (Heart Failure)
coefficient_female_hf = np.array([
    0.8998234868, -0.4559771121, 0.3576504886, 1.038346052, 0.5839160085, -0.0072293999,
    0.2997705936, 0.7451637983, 0.0557086989, 0.3534441888, -0.0981511027, -0.0946663022,
    -0.3581041098, -0.1159453019, -0.003878, -0.1884288937, -4.310409069
]).reshape(17, 1)

# Subset the data for females (female == 1) 
covariate_female_hf = df[(df['female'] == 1)][[
    'age', 'sbp1', 'sbp2', 'diabetes', 'cursmk', 'bmi1', 'bmi2', 'egfr1', 'egfr2',
    'antihtn', 'sbp2treat', 'agesbp2', 'agediabetes', 'agecursmk', 'agebmi2', 'ageegfr1', 'cons'
]].values

# Calculate the linear combination
lo_female_hf = np.dot(covariate_female_hf, coefficient_female_hf)

# Calculate the risk (probability)
risk_female_hf = np.exp(lo_female_hf) / (1 + np.exp(lo_female_hf))

# Evaluate Fairness

## ASCVD

In [ ]:
female_df = df[(df['female'] == 1)]
male_df = df[(df['female'] == 0)]

# Concatenate the DataFrames with females on top and males at the bottom
test_df = pd.concat([female_df, male_df], axis=0).reset_index(drop=True)

# Create female_test and male_test arrays
female_test = (test_df['female'] == 1.0).astype(int).values
male_test = (test_df['female'] == 0.0).astype(int).values

# Create s_ascvd_test array indicating whether an ASCVD event (either MI or stroke) occurred
s_ascvd_test = ((test_df['event_mi'] == 1) | (test_df['event_str'] == 1)).astype(int).values

# Create t_ascvd_test array with follow-up times for ASCVD
t_ascvd_test = test_df['fu_ascvd'].values

# Combine risk_female_ascvd and risk_male_ascvd into one array
combined_risk_ascvd = np.concatenate((risk_female_ascvd, risk_male_ascvd))
combined_risk_ascvd = combined_risk_ascvd.ravel()

In [ ]:
test_df.head(10)

### xCI 

In [ ]:
def xCI_spark(s_test, t_test, pred_risk,
              weights=None,
              pos_group=None, neg_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType
    
    # Initialize SparkSession if not already available
    spark = SparkSession.builder.getOrCreate()
    
    # Convert NumPy arrays to Python lists
    s_test_list = s_test.astype(int).tolist()
    t_test_list = t_test.astype(float).tolist()
    pred_risk_list = pred_risk.astype(float).tolist()
    
    # Handle weights if provided
    if weights is not None:
        weights_list = weights.astype(float).tolist()
    else:
        weights_list = [1.0] * len(s_test_list)
    
    # Convert pos_group and neg_group if provided
    if pos_group is not None:
        pos_group_list = pos_group.astype(bool).tolist()
    else:
        pos_group_list = [True] * len(s_test_list)
    
    if neg_group is not None:
        neg_group_list = neg_group.astype(bool).tolist()
    else:
        neg_group_list = [True] * len(s_test_list)
    
    # Create a list of tuples representing each row
    data = list(zip(s_test_list, t_test_list, pred_risk_list,
                    weights_list, pos_group_list, neg_group_list))
    
    # Define the schema explicitly
    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('s_test', IntegerType(), True),
        StructField('t_test', FloatType(), True),
        StructField('pred_risk', FloatType(), True),
        StructField('weights', FloatType(), True),
        StructField('pos_group', BooleanType(), True),
        StructField('neg_group', BooleanType(), True)
    ])
    
    # Add unique IDs to each row for joining purposes
    data_with_id = [(i,) + row for i, row in enumerate(data)]
    
    # Create the Spark DataFrame
    df = spark.createDataFrame(data_with_id, schema=schema)
    
    # Prepare the positive group DataFrame
    df_pos = df.filter((col('s_test') == 1) & (col('pos_group') == True))
    df_pos = df_pos.select(col('id').alias('id_pos'),
                           col('t_test').alias('t_test_pos'),
                           col('pred_risk').alias('pred_risk_pos'),
                           col('weights').alias('weights_pos'))
    
    # Prepare the negative group DataFrame
    df_neg = df.filter(col('neg_group') == True)
    df_neg = df_neg.select(col('id').alias('id_neg'),
                           col('t_test').alias('t_test_neg'),
                           col('pred_risk').alias('pred_risk_neg'),
                           col('weights').alias('weights_neg'))
    
    # Perform cross join between positive and negative groups
    df_pairs = df_pos.crossJoin(df_neg)
    
    # Apply the time condition
    df_valid = df_pairs.filter(col('t_test_pos') < col('t_test_neg'))
    
    # Compute weight product
    df_valid = df_valid.withColumn('weight_product', col('weights_pos') * col('weights_neg'))
    
    # Compute risk differences
    df_valid = df_valid.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_neg'))
    
    # Determine correctly ranked and tied
    df_valid = df_valid.withColumn(
        'correctly_ranked',
        when(col('risk_diff') > tied_tol, 1.0).otherwise(0.0)
    )
    df_valid = df_valid.withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    )
    
    # Compute numerator and denominator for concordance index
    df_valid = df_valid.withColumn('contribution',
                                   (col('correctly_ranked') + col('tied')) * col('weight_product'))
    
    numerator = df_valid.agg({'contribution': 'sum'}).collect()[0][0]
    denominator = df_valid.agg({'weight_product': 'sum'}).collect()[0][0]
    
    ci = numerator / denominator if denominator != 0 else None
    
    if return_num_valid:
        num_valid = df_valid.count()
        return ci, num_valid
    else:
        return ci


In [ ]:
xCI_spark(s_ascvd_test, t_ascvd_test, combined_risk_ascvd, return_num_valid=True)

In [ ]:
for g1, l1 in zip([female_test, male_test], ['female', 'male']):

    for g2, l2 in zip([female_test, male_test], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_ascvd_test, t_ascvd_test, combined_risk_ascvd,
                pos_group=g1, neg_group=g2
            )
        ))

## Heart Failure

In [ ]:
# Create s_ascvd_test array indicating whether an Heart Faliure(HF) occurred
s_hf_test = ((test_df['event_hf'] == 1)).astype(int).values

# Create t_ascvd_test array with follow-up times for HF
t_hf_test = test_df['fu_hf'].values

# Combine risk_female_ascvd and risk_male_ascvd into one array
combined_risk_hf = np.concatenate((risk_female_hf, risk_male_hf))
combined_risk_hf = combined_risk_hf.ravel()

In [ ]:
for g1, l1 in zip([female_test, male_test], ['female', 'male']):

    for g2, l2 in zip([female_test, male_test], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_hf_test, t_hf_test, combined_risk_hf,
                pos_group=g1, neg_group=g2
            )
        ))

In [ ]:
xCI_spark(s_hf_test, t_hf_test, combined_risk_hf, return_num_valid=True)

## CVD

In [ ]:
# Create s_cvd_test array indicating whether an CVD occurred
s_cvd_test = ((test_df['event_mi'] == 1) | (test_df['event_str'] == 1) |(test_df['event_hf'] == 1) ).astype(int).values

# Create t_cvd_test array with follow-up times for CVD
t_cvd_test = np.minimum(test_df['fu_ascvd'].values, test_df['fu_hf'].values)

# Combine risk_female_ascvd and risk_male_ascvd into one array
combined_risk_cvd = np.concatenate((risk_female_cvd, risk_male_cvd))
combined_risk_cvd = combined_risk_cvd.ravel()

In [ ]:
for g1, l1 in zip([female_test, male_test], ['female', 'male']):

    for g2, l2 in zip([female_test, male_test], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_cvd_test, t_cvd_test, combined_risk_cvd,
                pos_group=g1, neg_group=g2
            )
        ))

In [ ]:
xCI_spark(s_cvd_test, t_cvd_test, combined_risk_cvd, return_num_valid=True)

## Check Incidence Rate

In [ ]:
female_ascvd_incedence_rate = df[ (df['female'] == 1) & ((df['event_mi'] == 1)|(df['event_str'] == 1))].shape[0]/df[ (df['female'] == 1)].shape[0]
male_ascvd_incedence_rate = df[ (df['female'] == 0) & ((df['event_mi'] == 1)|(df['event_str'] == 1))].shape[0]/df[ (df['female'] == 0)].shape[0]
print(female_ascvd_incedence_rate)
print(male_ascvd_incedence_rate)

In [ ]:
female_hf_incedence_rate = df[ (df['female'] == 1) & (df['event_hf'] == 1)].shape[0]/df[ (df['female'] == 1)].shape[0]
male_hf_incedence_rate = df[ (df['female'] == 0) & (df['event_hf'] == 1)].shape[0]/df[ (df['female'] == 0)].shape[0]
print(female_hf_incedence_rate)
print(male_hf_incedence_rate)

In [ ]:
female_cvd_incedence_rate = df[ (df['female'] == 1) & ((df['event_mi'] == 1)|(df['event_str'] == 1)|(df['event_hf'] == 1))].shape[0]/df[ (df['female'] == 1)].shape[0]
male_cvd_incedence_rate = df[ (df['female'] == 0) & ((df['event_mi'] == 1)|(df['event_str'] == 1)|(df['event_hf'] == 1))].shape[0]/df[ (df['female'] == 0)].shape[0]
print(female_cvd_incedence_rate)
print(male_cvd_incedence_rate)

# Discrete Neural Network Model

In [ ]:
import torch

In [ ]:
print(saved_df.columns[10:25])

In [ ]:
def clean_and_prepare_data(df, random_state=0):
    """
    Cleans and preprocesses the DataFrame for further analysis or modeling.
    
    Args:
        df (pd.DataFrame): Input DataFrame containing the data to be cleaned.
        random_state (int): Random state for reproducibility when shuffling the data.
        
    Returns:
        tuple: features (numpy array), event_indicator (numpy array), event_time (numpy array)
    """
    # Define columns to include
    COLUMNS_TO_INCLUDE = ['age', 'female', 'scre', 'sbp', 'bmi', 'chol',
       'hdlc', 'diabetes', 'cursmk', 'antihtn', 'statin', 'event_mi', 'fu_mi']

    # Include only specified columns
    df = df[COLUMNS_TO_INCLUDE]
    
    # Fill missing values with the median
    df = df.apply(lambda col: col.fillna(col.median()), axis=0)
    
    # Standardize numeric columns
    # numeric_cols = df.select_dtypes(include=['float64']).columns
    # df[numeric_cols] = df[numeric_cols].transform(lambda x: (x - x.mean()) / x.std())
    
    # Shuffle the DataFrame
    df = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    # Extract features, event indicator, and event time
    features = df.drop(['event_mi', 'fu_mi'], axis=1).values.astype(float)
    event_indicator = df['event_mi'].values.astype(float)
    event_time = df['fu_mi'].values.astype(float)
    
    return features, event_indicator, event_time

In [ ]:
X, s, t = clean_and_prepare_data(saved_df)
test_idx = len(X) * 4 // 5

train_X, train_s, train_t = (arr[:test_idx] for arr in (X, s, t))
test_X, test_s, test_t = (arr[test_idx:] for arr in (X, s, t))

In [ ]:
class DiscreteTimeNN(torch.nn.Module):

    def __init__(self, hidden_layer_sizes, num_bins):
        super(DiscreteTimeNN, self).__init__()

        self.encoder_layers = [
            torch.nn.LazyLinear(size)
            for size in hidden_layer_sizes
        ]
        
        self.activation = torch.nn.ReLU()
        self.prediction_head = torch.nn.LazyLinear(num_bins + 1)
        self.softmax = torch.nn.Softmax(-1)

    def forward(self, x):
        
        for layer in self.encoder_layers:
            x = layer(x)
            x = self.activation(x)
            
        x = self.prediction_head(x)
        x = self.softmax(x)

        return x

In [ ]:
class DiscreteFailureTimeNLL(torch.nn.Module):
    
    def __init__(self, bin_boundaries, tolerance=1e-8):
        super(DiscreteFailureTimeNLL,self).__init__()
        
        self.bin_starts = torch.tensor(bin_boundaries[:-1])
        self.bin_ends = torch.tensor(bin_boundaries[1:])
        
        self.bin_lengths = self.bin_ends - self.bin_starts
        
        self.tolerance = tolerance
        
    def _discretize_times(self, times):

        return (
            (times[:, None] > self.bin_starts[None, :])
            & (times[:, None] <= self.bin_ends[None, :])
        )

    def _get_proportion_of_bins_completed(self, times):

        return torch.maximum(
            torch.minimum(
                (times[:, None] - self.bin_starts[None, :]) / self.bin_lengths[None, :],
                torch.tensor(1)
            ),
            torch.tensor(0)
        )
    
    def forward(self, predictions, event_indicators, event_times):

        event_likelihood = torch.sum(
            self._discretize_times(event_times) * predictions[:, :-1],
            -1
        ) + self.tolerance

        nonevent_likelihood = 1 - torch.sum(
            self._get_proportion_of_bins_completed(event_times) * predictions[:, :-1],
            -1
        ) + self.tolerance
        
        log_likelihood = event_indicators * torch.log(event_likelihood)
        log_likelihood += (1 - event_indicators) * torch.log(nonevent_likelihood)
        
        return -1. * torch.mean(log_likelihood)

In [ ]:
model = DiscreteTimeNN((100, ), 10)
loss_fn = DiscreteFailureTimeNLL(BIN_BOUNDARIES)
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
def get_batches(*arrs, batch_size=1):
    l = len(arrs[0])
    for ndx in range(0, l, batch_size):
        yield (torch.tensor(arr[ndx:min(ndx + batch_size, l)], dtype=torch.float) for arr in arrs)

In [ ]:
NUM_EPOCHS = 70

train_loss = []
test_loss = []

for epoch_idx in range(NUM_EPOCHS):
    
    epoch_train_loss = []
    epoch_test_loss = []

    for batch_X, batch_s, batch_t in get_batches(train_X, train_s, train_t, batch_size=100):

        optimizer.zero_grad()

        # Make predictions for this batch
        batch_predictions = model(batch_X)
        
        # Compute the loss and its gradients
        loss = loss_fn(batch_predictions, batch_s, batch_t)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # Gather data and report
        epoch_train_loss.append(loss.item())
        
    for batch_X, batch_s, batch_t in get_batches(test_X, test_s, test_t, batch_size=100):

        # Make predictions for this batch
        batch_predictions = model(batch_X)
        
        # Compute the loss and its gradients
        loss = loss_fn(batch_predictions, batch_s, batch_t)
        
        # Gather data and report
        epoch_test_loss.append(loss.item())
        
    train_loss.append(np.mean(epoch_train_loss))
    test_loss.append(np.mean(epoch_test_loss))
            
    print('Completed epoch %i; train loss = %.3f; test loss = %.3f' % (
        epoch_idx, train_loss[-1], test_loss[-1]), end='\r')
          
plt.plot(np.arange(len(train_loss)), train_loss, label='Training Loss')
plt.plot(np.arange(len(test_loss)), test_loss, label='Test Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss (NLL)')
plt.legend()
plt.show()

In [ ]:
test_predictions = (
    model(torch.tensor(test_X, dtype=torch.float))
    .detach()
    .numpy()
)

pred_mean = np.mean(test_predictions, axis=0)
# pred_lower = np.percentile(test_predictions, 2.5, axis=0)
# pred_upper = np.percentile(test_predictions, 97.5, axis=0)

plt.bar(
    np.arange(11),
    np.mean(test_predictions, axis=0),
    # yerr=np.stack([
    #     pred_mean - pred_lower,
    #     pred_upper - pred_mean
    # ])
)

plt.xlabel('Bin')
plt.ylabel('Predicted Event Probability')
plt.show()

In [ ]:
print(test_predictions.shape)

In [ ]:
print(test_s.shape)

In [ ]:
def main():

    df_train = pd.read_csv('datasets/support2_train_outcomes.csv')
    df_test = pd.read_csv('datasets/support2_test_predictions.csv')

    s_train = df_train['death'].values
    s_test = df_test['death'].values

    t_train = df_train['time'].values
    t_test = df_test['time'].values

    pred_risk = df_test['pred_risk'].values

    print('CI = %.3f' % xCI(s_test, t_test, pred_risk))
    
    print('IPCW CI = %.3f' % xCI(
        s_test, t_test, pred_risk,
        ipcw=True, s_train=s_train, t_train=t_train
    ))


def hazard_components(s_true, t_true):

    df = (
        pd.DataFrame({'event': s_true, 'time': t_true})
        .groupby('time')
        .agg(['count', 'sum'])
    )

    t = df.index.values
    d = df[('event', 'sum')].values
    c = df[('event', 'count')].values
    n = np.sum(c) - np.cumsum(c) + c

    return t, d, n


def kaplan_meier(s_true, t_true, t_new=None):

    t, d, n = hazard_components(s_true, t_true)

    m = np.cumprod(1 - np.divide(
        d, n,
        out=np.zeros(len(d)),
        where=n > 0
    ))
    
    v = (m ** 2) * np.cumsum(np.divide(
        d, n * (n - d),
        out=np.zeros(len(d)),
        where=n * (n - d) > 0
    ))

    if t_new is not None:
        return interpolate(t, m, t_new)
    else:
        return t, m, v


def nelson_aalen(s_true, t_true, t_new=None):

    t, d, n  = hazard_components(s_true, t_true)

    m = np.cumsum(np.divide(
        d, n,
        out=np.zeros(len(d)),
        where=n > 0
    ))
    
    v = np.cumsum(np.divide(
        (n - d) * d,
        (n - 1) * (n ** 2),
        out=np.zeros(len(d)),
        where=(n - 1) > 0
    ))
        
    if t_new is not None:
        return interpolate(t, m, t_new)
    else:
        return t, m, v


def interpolate(x, y, new_x, method='pad'):
    
    # s = pd.Series(data=y, index=x)
    # new_y = (s
    #     .reindex(s.index.union(new_x).unique())
    #     .interpolate(method=method)[new_x]
    #     .values
    # )
    
    # return new_y

    return np.interp(new_x, x, y)


def bootstrappable(func):

    def bootstrap_func(*args, n_bootstrap_samples=None, random_state=None, ci=95, **kwargs):

        estimate = func(*args, **kwargs)

        if n_bootstrap_samples is not None:

            assert type(n_bootstrap_samples) is int, n_bootstrap_samples

            rs = np.random.RandomState(seed=random_state)
            
            N = len(args[0])
            indices = np.arange(N)

            samples = []

            for _ in range(n_bootstrap_samples):

                sample_indices = rs.choice(indices, len(indices), replace=True)

                func_args = [
                    arg[sample_indices]
                    if (hasattr(arg, '__len__') and (len(arg) == N))
                    else arg
                    for arg in args
                ]
                
                func_kwargs = {
                    kw: arg[sample_indices]
                    if (hasattr(arg, '__len__') and (len(arg) == N))
                    else arg
                    for kw, arg in kwargs.items()
                }

                samples.append(func(*func_args, **func_kwargs))

            return (
                estimate,
                np.percentile(samples, 50 - ci / 2., axis=0),
                np.percentile(samples, 50 + ci / 2., axis=0)
            )

        else:

            return estimate

    return bootstrap_func


@bootstrappable
def xAUCt(s_test, t_test, pred_risk, times, pos_group=None, neg_group=None):

    # NOTE: enter groups pos_group and neg_group for xAUC_t; omit for AUC_t

    # pred_risk can be 1d (static) or 2d (time-varying)
    if len(pred_risk.shape) == 1:
        pred_risk = pred_risk[:, np.newaxis]

    # positives: s_test = 1 & t_test =< t
    pos = (t_test[:, np.newaxis] <= times[np.newaxis, :]) & s_test[:, np.newaxis]

    if pos_group is not None:
        pos = pos & pos_group[:, np.newaxis]
    
    # negatives: t_test > t
    neg = (t_test[:, np.newaxis] > times[np.newaxis, :])

    if neg_group is not None:
        neg = neg & neg_group[:, np.newaxis]

    valid = pos[:, np.newaxis, :] & neg[np.newaxis, :, :]
    correctly_ranked = valid & (pred_risk[:, np.newaxis, :] > pred_risk[np.newaxis, :, :])

    return np.sum(correctly_ranked, axis=(0, 1)) / np.sum(valid, axis=(0, 1))


@bootstrappable
def xAPt(s_test, t_test, pred_risk, times, pos_group=None, neg_group=None, return_prevalence=False):

    ap = []
    prev = []

    for idx, time in enumerate(times):

        # pred_risk can be 1d (static) or 2d (time-varying)
        if len(pred_risk.shape) == 1:
            prt = pred_risk
        else:
            prt = pred_risk[:, idx]

        recall, precision, threshold, prevalence = xPRt(
            s_test, t_test, prt, time,
            pos_group=pos_group, neg_group=neg_group
        )
        
        ap.append(-1 * np.sum(np.diff(recall) * np.array(precision)[:-1]))
        prev.append(prevalence)

    return (np.array(ap), np.array(prev)) if return_prevalence else np.array(ap)


def xROCt(s_test, t_test, pred_risk, time, pos_group=None, neg_group=None):

    # NOTE: enter groups pos_group and neg_group for xROC_t; omit for ROC_t

    threshold = np.append(np.sort(pred_risk), np.infty)

    # positives: s_test = 1 & t_test =< t
    pos = (t_test < time) & s_test

    if pos_group is not None:
        pos = pos & pos_group
    
    # negatives: t_test > t
    neg = (t_test > time)

    if neg_group is not None:
        neg = neg & neg_group

    # prediction
    pred = pred_risk[:, np.newaxis] > threshold[np.newaxis, :]

    tpr = np.sum(pred & pos[:, np.newaxis], axis=0) / np.sum(pos)
    fpr = np.sum(pred & neg[:, np.newaxis], axis=0) / np.sum(neg)

    return tpr, fpr, threshold


def xPRt(s_test, t_test, pred_risk, time, pos_group=None, neg_group=None):

    threshold = np.append(np.sort(pred_risk), np.infty)

    # positives: s_test = 1 & t_test =< t
    pos = (t_test < time) & s_test

    if pos_group is not None:
        pos = pos & pos_group
    
    # negatives: t_test > t
    neg = (t_test > time)

    if neg_group is not None:
        neg = neg & neg_group

    # prediction
    pred = pred_risk[:, np.newaxis] > threshold[np.newaxis, :]

    tps = np.sum(pred & pos[:, np.newaxis], axis=0)
    fps = np.sum(pred & neg[:, np.newaxis], axis=0)

    positives = np.sum(pos)
    negatives = np.sum(neg)

    recall = tps / positives
    precision = np.divide(tps, tps + fps, out=np.ones_like(tps, dtype=float), where=(tps + fps) > 0)

    prevalence = positives / (positives + negatives)

    return recall, precision, threshold, prevalence


def ipc_weights(s_train, t_train, s_test, t_test, tau=None):

    if tau == 'auto':
        mask = t_test < t_train[s_train == 1].max()
        #mask = t_test < t_train[s_train == 0].max()
    
    elif tau is not None:
        mask = t_test < tau

    else:
        mask = np.ones_like(t_test, dtype=bool)

    pc = kaplan_meier(1 - s_train, t_train, t_test)
    pc[s_test == 0] = 1.

    w = 1. / pc
    w[~mask] = 0.

    return w


@bootstrappable
def xCI(s_test, t_test, pred_risk,
        weights=None,
        pos_group=None, neg_group=None,
        return_num_valid=False,
        tied_tol=1e-8):

    w = weights if weights is not None else np.ones_like(s_test)

    mask1 = (s_test == 1)

    if pos_group is not None:
        mask1 = mask1 & pos_group

    w = w[mask1, np.newaxis]

    mask2 = np.ones_like(s_test, dtype=bool)

    if neg_group is not None:
        mask2 = mask2 & neg_group

    valid = t_test[mask1, np.newaxis] < t_test[np.newaxis, mask2]

    risk_diff = pred_risk[mask1, np.newaxis] - pred_risk[np.newaxis, mask2]

    correctly_ranked = valid & (risk_diff > tied_tol)
    tied = valid & (np.abs(risk_diff) <= tied_tol)

    num_valid = np.sum((w ** 2) * valid)
    ci = np.sum((w ** 2) * (correctly_ranked + 0.5 * tied)) / num_valid

    return (ci, num_valid) if return_num_valid else ci


@bootstrappable
def xxCI(s_test, t_test, pred_risk, weights=None, pos_group=None, neg_group=None):

    m1, n1 = xCI(
        s_test, t_test, pred_risk,
        weights=weights,
        pos_group=pos_group, neg_group=neg_group,
        return_num_valid=True
    )
    
    m2, n2 = xCI(
        s_test, t_test, pred_risk,
        weights=weights,
        pos_group=neg_group, neg_group=pos_group,
        return_num_valid=True
    )
    
    return (m1 * n1 + m2 * n2) / (n1 + n2)



In [ ]:
times = BIN_BOUNDARIES[1:]

cumulative_predicted_risk = np.cumsum(test_predictions, axis=1)[:, :-1]

cp_mean = [0] + list(cumulative_predicted_risk.mean(axis=0))
cp_std = [0] + list(cumulative_predicted_risk.std(axis=0))

fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)

km_times, km_mean, km_var = kaplan_meier(test_s, test_t)

ax[0].plot(km_times, 1 - km_mean)
ax[0].fill_between(km_times, 1 - km_mean - np.sqrt(km_var), 1 - km_mean + np.sqrt(km_var), alpha=.3)

ax[1].plot([0] + list(times), cp_mean)
ax[1].fill_between(
    [0] + list(times),
    np.array(cp_mean) - np.array(cp_std),
    np.array(cp_mean) + np.array(cp_std),
    alpha=.3
)

plt.show()

In [ ]:
xCI_spark(test_s, test_t, cumulative_predicted_risk[:, 1], return_num_valid=True)

In [ ]:
n_bootstrap_samples = 1

fig, ax = plt.subplots(1, 4, figsize=(16, 4))

times = BIN_BOUNDARIES[1:]
time_idx = 6
time = times[time_idx]

cumulative_predicted_risk = np.cumsum(test_predictions, axis=1)[:, :-1]

auct, ci_low, ci_high = xAUCt(
    test_s == 1, test_t, cumulative_predicted_risk, times,
    n_bootstrap_samples=n_bootstrap_samples
)

ax[0].plot(times, auct, 'k-', label='All-All')
ax[0].fill_between(times, ci_low, ci_high, color='k', alpha=.1)

tprt, fprt, _ = xROCt(
    test_s == 1, test_t, cumulative_predicted_risk[:, time_idx], time
)

ax[1].plot(
    fprt, tprt, 'k-',
    label='AUC$_t$ = %.2f (%.2f-%.2f)' % (
        auct[time_idx], ci_low[time_idx], ci_high[time_idx]
    )
)

(apt, prevt), (apt_low, prev_low), (apt_high, prev_high) = xAPt(
    test_s == 1, test_t, cumulative_predicted_risk, times,
    return_prevalence=True,
    n_bootstrap_samples=n_bootstrap_samples
)

ax[2].plot(times, apt, 'k-', label='All')
ax[2].fill_between(times, apt_low, apt_high, color='k', alpha=.1)
ax[2].plot(times, prevt, 'k--')#, label='Prevalence at t')

recallt, precisiont, _, prevt = xPRt(
    test_s == 1, test_t, cumulative_predicted_risk[:, time_idx], time
)

ax[3].plot(
    recallt, precisiont, 'k-',
    label='AP$_t$ = %.2f (%.2f-%.2f)' % (
        apt[time_idx], apt_low[time_idx], apt_high[time_idx]
    )
)

ax[3].plot([0, 1], [prevt, prevt], 'k--')#, label='Prevalence at t')

ax[0].plot(times, times * 0 + .5, 'k--')
ax[0].set_ylim([0.45, 1.])
ax[0].set_ylabel('AUC$_t$')
ax[0].set_xlabel('Time (t)')
ax[0].legend()

ax[1].plot([0, 1], [0, 1], 'k--')
ax[1].set_ylabel('True Positive Rate (t=%i)' % time)
ax[1].set_xlabel('False Positive Rate (t=%i)' % time)
ax[1].legend()

ax[2].set_ylabel('AP$_t$')
ax[2].set_xlabel('Time (t)')
ax[2].legend()

ax[3].set_ylabel('Precision (t=%i)' % time)
ax[3].set_xlabel('Recall (t=%i)' % time)
ax[3].legend()

plt.tight_layout()
plt.show()